# S6E5 — Tuning dos Modelos

## Setup

### Imports e configuração

In [1]:
from pathlib import Path

import pandas as pd
import optuna
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

from mltemplate.config import ProjectConfig
from mltemplate.storage import StorageManager
from mltemplate.data import KaggleSource, DataManager
from mltemplate.tuning import OptunaTuner, XGBoostAdapter, LightGBMAdapter, KerasAdapter

import logging
logging.basicConfig(level=logging.INFO)

d:\data_science\kaggle_competitions\S6E5-predicting_f1_pit_stops\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
config = ProjectConfig(
    target="PitNextLap",                                                                                                                                                                   # ajuste para o nome real da coluna alvo
    numerical_features=["Year", "LapNumber", "Stint", "TyreLife", "Position", "LapTime (s)", "LapTime_Delta", "Cumulative_Degradation", "RaceProgress", "Position_Change"],                # preencha após ver o dataset
    categorical_features=["Compound", "Race", "PitStop"],                                                                                                                                  # preencha após ver o dataset
    ignore_features=["id", "Driver"],                                                                                                                                                                # ajuste se necessário
    problem_type="classification",
)

storage = StorageManager(root=Path("."))
dm      = DataManager(storage, config)

### Carregar feature set

In [3]:
X_train_v1, X_test_v1, y_train_v1, X_val_v1, y_val_v1, y_test_v1 = dm.load_feature_set(name="v1")

print(f"X_train_v1: {X_train_v1.shape}")
print(f"X_val_v1:   {X_val_v1.shape}")
print(f"X_test_v1:  {X_test_v1.shape}")

INFO:mltemplate.data.manager:Feature set 'v1' carregado — treino (351312, 42), teste (188165, 42)


X_train_v1: (351312, 42)
X_val_v1:   (87828, 42)
X_test_v1:  (188165, 42)


## v1 — XGBoost

### Tuning

In [4]:
def xgb_param_space_v1(trial: optuna.Trial) -> dict:
    return {
        "n_estimators":     trial.suggest_int("n_estimators", 100, 1000),
        "max_depth":        trial.suggest_int("max_depth", 3, 10),
        "learning_rate":    trial.suggest_float("learning_rate", 1e-3, 0.3, log=True),
        "subsample":        trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_alpha":        trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda":       trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
    }

result_xgb_v1 = OptunaTuner(config).tune(
    XGBoostAdapter(XGBClassifier),
    X_train_v1, y_train_v1,
    param_space_func=xgb_param_space_v1,
    scoring="roc_auc",
    X_val=X_val_v1, y_val=y_val_v1,
    n_trials=50,
)

print(f"XGB — ROC-AUC (val): {result_xgb_v1.score:.4f}")
print(f"Params: {result_xgb_v1.params}")

INFO:mltemplate.tuning.tuner:OptunaTuner — modo holdout, scoring=roc_auc, n_trials=50
Melhor: 0.9492: 100%|██████████| 50/50 [28:17<00:00, 33.94s/it]


XGB — ROC-AUC (val): 0.9492
Params: {'n_estimators': 1000, 'max_depth': 9, 'learning_rate': 0.023617996501521653, 'subsample': 0.6926714770728912, 'colsample_bytree': 0.7356120922972055, 'reg_alpha': 0.32374944427515806, 'reg_lambda': 1.7179459468039832e-07, 'min_child_weight': 1}


### Salvar

In [ ]:
storage.save_model(result_xgb_v1.model, "xgb_v1")
storage.save_metrics({"roc_auc": result_xgb_v1.score, "params": result_xgb_v1.params}, "xgb_v1")
print("Modelo e métricas XGB salvos.")

### Submissão

In [ ]:
proba_preds = result_xgb_v1.model.predict_proba(X_test_v1)[:, 1]
_, test_df  = dm.load_raw(KaggleSource("playground-series-s6e5"))
submission  = pd.DataFrame({"id": test_df["id"], config.target: proba_preds})
storage.save_submission(submission, "xgb_v1")
print(f"Submissão salva em: {storage.submissions_path / 'xgb_v1.csv'}")
submission.head()

## v1 — LightGBM

### Tuning

In [ ]:
def lgbm_param_space_v1(trial: optuna.Trial) -> dict:
    return {
        "n_estimators":      trial.suggest_int("n_estimators", 100, 1000),
        "max_depth":         trial.suggest_int("max_depth", 3, 10),
        "learning_rate":     trial.suggest_float("learning_rate", 1e-3, 0.3, log=True),
        "num_leaves":        trial.suggest_int("num_leaves", 20, 300),
        "subsample":         trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree":  trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_alpha":         trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda":        trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),
    }

result_lgbm_v1 = OptunaTuner(config).tune(
    LightGBMAdapter(LGBMClassifier),
    X_train_v1, y_train_v1,
    param_space_func=lgbm_param_space_v1,
    scoring="roc_auc",
    X_val=X_val_v1, y_val=y_val_v1,
    n_trials=100,
)

print(f"LGBM — ROC-AUC (val): {result_lgbm_v1.score:.4f}")
print(f"Params: {result_lgbm_v1.params}")

INFO:mltemplate.tuning.tuner:OptunaTuner — modo holdout, scoring=roc_auc, n_trials=50
Melhor: 0.9411:   4%|▍         | 2/50 [00:30<12:54, 16.13s/it]

### Salvar

In [ ]:
storage.save_model(result_lgbm_v1.model, "lgbm_v1")
storage.save_metrics({"roc_auc": result_lgbm_v1.score, "params": result_lgbm_v1.params}, "lgbm_v1")
print("Modelo e métricas LGBM salvos.")

### Submissão

In [ ]:
proba_preds = result_lgbm_v1.model.predict_proba(X_test_v1)[:, 1]
_, test_df  = dm.load_raw(KaggleSource("playground-series-s6e5"))
submission  = pd.DataFrame({"id": test_df["id"], config.target: proba_preds})
storage.save_submission(submission, "lgbm_v1")
print(f"Submissão salva em: {storage.submissions_path / 'lgbm_v1.csv'}")
submission.head()

## v1 — Keras

### Tuning

In [4]:
def keras_param_space_v1(trial: optuna.Trial) -> dict:
    return {
        "n_layers":      trial.suggest_int("n_layers", 1, 4),
        "hidden_dim":    trial.suggest_categorical("hidden_dim", [64, 128, 256, 512]),
        "dropout":       trial.suggest_float("dropout", 0.0, 0.5),
        "activation":    trial.suggest_categorical("activation", ["relu"]),
        "learning_rate": trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True),
        "batch_size":    trial.suggest_categorical("batch_size", [256, 512, 1024]),
        "epochs":        100,
        "patience":      10,
    }

result_keras_v1 = OptunaTuner(config).tune(
    KerasAdapter(config),
    X_train_v1, y_train_v1,
    param_space_func=keras_param_space_v1,
    scoring="roc_auc",
    X_val=X_val_v1, y_val=y_val_v1,
    n_trials=10,
)

print(f"Keras — ROC-AUC (val): {result_keras_v1.score:.4f}")
print(f"Params: {result_keras_v1.params}")

INFO:mltemplate.tuning.tuner:OptunaTuner — modo holdout, scoring=roc_auc, n_trials=10
Melhor: 0.9441: 100%|██████████| 10/10 [43:52<00:00, 263.23s/it]


Keras — ROC-AUC (val): 0.9441
Params: {'n_layers': 2, 'hidden_dim': 256, 'dropout': 0.28059809303281247, 'activation': 'relu', 'learning_rate': 0.001014299778430435, 'batch_size': 1024, 'epochs': 100, 'patience': 10}


### Salvar

In [5]:
storage.save_model(result_keras_v1.model, "keras_v1")
storage.save_metrics({"roc_auc": result_keras_v1.score, "params": result_keras_v1.params}, "keras_v1")
print("Modelo e métricas Keras salvos.")

INFO:mltemplate.storage:Modelo salvo em models\keras_v1.pkl
INFO:mltemplate.storage:Métricas salvas em reports\metrics\keras_v1.json


Modelo e métricas Keras salvos.


### Submissão

In [6]:
proba_preds = result_keras_v1.model.predict_proba(X_test_v1)[:, 1]
_, test_df  = dm.load_raw(KaggleSource("playground-series-s6e5"))
submission  = pd.DataFrame({"id": test_df["id"], config.target: proba_preds})
storage.save_submission(submission, "keras_v1")
print(f"Submissão salva em: {storage.submissions_path / 'keras_v1.csv'}")
submission.head()

INFO:mltemplate.data.sources:Dados já existentes em data\raw — download ignorado.
INFO:mltemplate.data.manager:Dados brutos carregados — treino (439140, 16), teste (188165, 15)


Submissão salva em: submissions\keras_v1.csv


,id,PitNextLap
0,439140,0.012580
1,439141,0.032584
2,439142,0.012319
3,439143,0.231929
4,439144,0.818661
